# 03 - Feature Engineering
**Project:** Air Quality & Pollution Intelligence - Data Mining and Business Intelligence

**Objective:** Create the calendar, seasonal and pollution-composition features that the analysis and the dashboard need, while reusing the supplied AQI instead of inventing one.

**How to read this notebook:** every number printed below is produced by the
code in this notebook from `data/raw/Air_quality_data.csv`. Column names are
discovered at runtime through `src/config.py`, so nothing is assumed.


In [1]:
"""Environment bootstrap: make src/ importable and pin the working directory."""
import sys, os, warnings
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
os.chdir(PROJECT_ROOT)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 90)
%matplotlib inline
print("project root:", PROJECT_ROOT)

project root: C:\Users\DELL\OneDrive\Desktop\Air Quality


In [2]:
import config as C
import data_utils as U
import preprocessing as P
import feature_engineering as FE

clean, log, *_ = P.clean(U.load_raw())
feat, notes = FE.build_features(clean)
print("columns after engineering:", feat.shape[1])
for n in notes:
    print(" -", n)

columns after engineering: 47
 - Calendar features added: Year, Month, Month_Name, Quarter, Day, Day_of_Week, Week_of_Year, Is_Weekend, Year_Month
 - Season added using the Indian meteorological convention Winter=12-2, Summer=3-5, Monsoon=6-9, Post-Monsoon=10-11 (project-specific, not a universal scientific definition)
 - Pollution_Index built from 9 pollutants (PM2.5, PM10, NO, NO2, NOx, NH3, CO, SO2, O3); project-specific scale, not official AQI
 - Per-city lag/rolling features added for trend readability


## 1. Calendar features

`Year`, `Month`, `Month_Name`, `Quarter`, `Day`, `Day_of_Week`, `Week_of_Year`,
`Is_Weekend`, `Year_Month`. These let the BI dashboard slice the same measure by
time period without re-aggregating the raw text dates.

In [3]:
cal = ["Year", "Month", "Month_Name", "Quarter_Label", "Day", "Day_Name",
       "Week_of_Year", "Is_Weekend", "Season", "Year_Month"]
display(feat[["Date"] + cal].head(8))
print("date span :", feat["Date"].min().date(), "->", feat["Date"].max().date())
print("years     :", sorted(feat["Year"].unique()))
print("sanity check, month 1 ->", feat.loc[feat["Month"] == 1, "Month_Name"].unique()[0])
print("sanity check, weekday 0 ->", feat.loc[feat["Day_of_Week"] == 0, "Day_Name"].unique()[0])

,Date,Year,Month,Month_Name,Quarter_Label,Day,Day_Name,Week_of_Year,Is_Weekend,Season,Year_Month
4,2015-01-01,2015,1,January,Q1,1,Thursday,1,0,Winter,2015-01
9,2015-01-02,2015,1,January,Q1,2,Friday,1,0,Winter,2015-01
14,2015-01-03,2015,1,January,Q1,3,Saturday,1,1,Winter,2015-01
19,2015-01-04,2015,1,January,Q1,4,Sunday,1,1,Winter,2015-01
24,2015-01-05,2015,1,January,Q1,5,Monday,2,0,Winter,2015-01
29,2015-01-06,2015,1,January,Q1,6,Tuesday,2,0,Winter,2015-01
34,2015-01-07,2015,1,January,Q1,7,Wednesday,2,0,Winter,2015-01
39,2015-01-08,2015,1,January,Q1,8,Thursday,2,0,Winter,2015-01


date span : 2015-01-01 -> 2024-12-31
years     : [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
sanity check, month 1 -> January
sanity check, weekday 0 -> Monday


## 2. Season definition (documented, not universal)

Winter = Dec-Feb, Summer = Mar-May, Monsoon = Jun-Sep, Post-Monsoon = Oct-Nov.
This is the convention normally used in Indian climate reporting and is adopted
here as a *project* definition. It is not presented as a scientifically
universal partition of the year.

In [4]:
season_check = feat.groupby("Season", observed=True).agg(
    months=("Month", lambda s: sorted(s.unique())), records=("Season", "size"))
display(season_check)
print("C. config.SEASON_MAP =", C.SEASON_MAP)

,months,records
Season,,
Winter,"[1, 2, 3]",4515
Summer,"[4, 5, 6]",4550
Monsoon,"[7, 8, 9]",4600
Post-Monsoon,"[10, 11, 12]",4600


C. config.SEASON_MAP = {1: 'Winter', 2: 'Winter', 3: 'Winter', 4: 'Summer', 5: 'Summer', 6: 'Summer', 7: 'Monsoon', 8: 'Monsoon', 9: 'Monsoon', 10: 'Post-Monsoon', 11: 'Post-Monsoon', 12: 'Post-Monsoon'}


## 3. Pollution-composition features

| Feature | Definition | Purpose |
|---|---|---|
| `PM25_PM10_Ratio` | PM2.5 / PM10 | fine vs coarse particulate share |
| `NO2_NOx_Ratio` | NO2 / NOx | aged vs freshly emitted nitrogen oxides |
| `O3_PM10_Ratio` | O3 / PM10 | secondary vs primary pollution balance |
| `Pollution_Index` | mean of min-max-scaled pollutants, x100 | one common 0-100 scale |
| `Pollutants_Above_City_Q3` | count of pollutants above that city's own 75th percentile | how many pollutants are simultaneously high |
| `City_Pollutant_Percentile` | mean within-city percentile across pollutants | comparable across cities |
| `AQI_Lag_1`, `AQI_Change_1d`, `AQI_Rolling_7/30` | per-city time features | smoothed trend lines |

**Important:** `Pollution_Index` is a project-specific analytical index. The
official AQI already exists in the data and is used unchanged everywhere else.

In [5]:
new_cols = ["PM25_PM10_Ratio", "NO2_NOx_Ratio", "O3_PM10_Ratio",
            "Pollution_Index", "Pollution_Index_Band", "Pollutants_Above_City_Q3",
            "City_Pollutant_Percentile", "AQI_Lag_1", "AQI_Change_1d",
            "AQI_Rolling_7", "AQI_Rolling_30"]
missing = [c for c in new_cols if c not in feat.columns]
print("expected engineered columns that are absent:", missing or "none")
display(feat[[C.CITY_COL, "Date"] + new_cols].head(6))

expected engineered columns that are absent: none


,City,Date,PM25_PM10_Ratio,NO2_NOx_Ratio,O3_PM10_Ratio,Pollution_Index,Pollution_Index_Band,Pollutants_Above_City_Q3,City_Pollutant_Percentile,AQI_Lag_1,AQI_Change_1d,AQI_Rolling_7,AQI_Rolling_30
4,Bangalore,2015-01-01,1.457944,0.200323,1.395922,46.94,Medium,3,47.1,NaN,NaN,339.800000,339.800000
9,Bangalore,2015-01-02,0.070866,0.476700,0.156850,45.65,Medium,2,45.7,339.8,-72.3,303.650000,303.650000
14,Bangalore,2015-01-03,3.852647,0.462500,1.708155,60.81,High,2,61.0,267.5,151.8,342.200000,342.200000
19,Bangalore,2015-01-04,0.189719,2.720670,0.368955,55.32,Medium,2,55.2,419.3,-173.6,318.075000,318.075000
24,Bangalore,2015-01-05,0.821551,0.843750,0.044223,33.41,Low,0,33.4,245.7,168.1,337.220000,337.220000
29,Bangalore,2015-01-06,0.863192,12.430769,0.225884,52.30,Medium,4,52.1,413.8,-294.9,300.833333,300.833333


In [6]:
feat[["Pollution_Index", "PM25_PM10_Ratio", "NO2_NOx_Ratio",
      "Pollutants_Above_City_Q3", "City_Pollutant_Percentile"]].describe().T

,count,mean,std,min,25%,50%,75%,max
Pollution_Index,18265.0,50.125958,9.600594,16.06,43.520000,50.170000,56.720000,87.05
PM25_PM10_Ratio,18263.0,4.086671,55.602870,0.00,0.416652,0.843983,1.689148,4042.00
NO2_NOx_Ratio,18264.0,2.520396,22.425480,0.00,0.300297,0.595090,1.193655,1448.00
Pollutants_Above_City_Q3,18265.0,2.246647,1.301719,0.00,1.000000,2.000000,3.000000,8.00
City_Pollutant_Percentile,18265.0,50.013315,9.586927,16.20,43.500000,50.100000,56.600000,86.90


### Guarded division

Ratios divide by a column that can be zero (PM10 = 0 exists in the data). The
implementation returns `NaN` instead of `inf` in that case; the count is shown
below so the reader knows how many records simply cannot have that ratio.

In [7]:
for r in ["PM25_PM10_Ratio", "NO2_NOx_Ratio", "O3_PM10_Ratio"]:
    print(f"{r}: {int(feat[r].isna().sum())} records with a zero/missing denominator")

PM25_PM10_Ratio: 2 records with a zero/missing denominator
NO2_NOx_Ratio: 1 records with a zero/missing denominator
O3_PM10_Ratio: 2 records with a zero/missing denominator


## 4. Feature inventory (saved for the report)

In [8]:
inv = FE.feature_inventory(feat)
display(inv)
inv.to_csv(C.PROCESSED_DIR / "feature_inventory.csv", index=False)
feat.to_csv(C.PROCESSED_DIR / "features_stage03.csv", index=False)
print("saved: feature_inventory.csv, features_stage03.csv")

,Feature,Present,Dtype,Purpose
0,Year,True,Int64,yearly trend comparison
1,Month,True,Int64,monthly seasonality
2,Month_Name,True,string,supporting BI attribute
3,Quarter,True,Int64,supporting BI attribute
4,Quarter_Label,True,str,supporting BI attribute
5,Day,True,Int64,supporting BI attribute
6,Day_of_Week,True,Int64,supporting BI attribute
7,Day_Name,True,string,supporting BI attribute
8,Week_of_Year,True,Int64,supporting BI attribute
9,Is_Weekend,True,int64,weekday vs weekend behaviour


saved: feature_inventory.csv, features_stage03.csv


## 5. Leakage guard for the engineered columns

A derived column that contains the target makes evaluation meaningless. Below,
the correlation of each engineered numeric feature with the AQI is shown; any
feature that is essentially a re-labelled AQI is excluded from modelling.

In [9]:
import statistics_analysis as S
eng = [c for c in ["Pollution_Index", "PM25_PM10_Ratio", "NO2_NOx_Ratio",
                   "O3_PM10_Ratio", "Pollutants_Above_City_Q3",
                   "City_Pollutant_Percentile", "Is_Weekend", "Month"]
       if c in feat.columns]
tab = S.correlation_with_target(feat, eng, C.AQI_COL)
display(tab)
print("Modelling in notebooks 06-08 uses only the measured pollutant columns,")
print("so engineered columns cannot leak the target into a model.")

,Variable,n,Pearson_r,Pearson_p,Spearman_rho,Spearman_p,r^2,Strength,Direction,Significant_at_0.05
0,Pollution_Index,18265,0.1063,4.540000e-47,0.1058,1.310000e-46,0.0113,weak,positive,True
1,City_Pollutant_Percentile,18265,0.1059,1.080000e-46,0.1053,3.160000e-46,0.0112,weak,positive,True
2,Pollutants_Above_City_Q3,18265,-0.0555,5.830000e-14,-0.0494,2.340000e-11,0.0031,negligible,negative,True
3,O3_PM10_Ratio,18263,-0.0279,1.650000e-04,-0.1427,1.080000e-83,0.0008,negligible,negative,True
4,PM25_PM10_Ratio,18263,-0.0268,2.960000e-04,-0.0560,3.660000e-14,0.0007,negligible,negative,True
5,Is_Weekend,18265,-0.0104,1.620000e-01,-0.0113,1.280000e-01,0.0001,negligible,negative,False
6,NO2_NOx_Ratio,18264,-0.0032,6.700000e-01,0.0126,8.790000e-02,0.0000,negligible,negative,False
7,Month,18265,0.0002,9.770000e-01,0.0008,9.190000e-01,0.0000,negligible,positive,False


Modelling in notebooks 06-08 uses only the measured pollutant columns,
so engineered columns cannot leak the target into a model.


## Verification checklist

- [x] Every date feature derived from a parsed datetime, spot-checked against the calendar
- [x] Season definition written down and labelled project-specific
- [x] Supplied AQI reused; derived index explicitly labelled as non-official
- [x] Division-by-zero handled and the affected record count reported
- [x] Engineered columns checked for target leakage

**Next:** `04_eda.ipynb`.